# Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import re
import yfinance as yf

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from scipy import stats

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("deep")

from langdetect import detect

print("Όλες οι βιβλιοθήκες φορτώθηκαν επιτυχώς!")

Όλες οι βιβλιοθήκες φορτώθηκαν επιτυχώς!


In [2]:
# ============================================================
# CELL 2: Φόρτωση & Πρώτη Εξερεύνηση Dataset
# ============================================================
# Στόχος: Να κατανοήσουμε τη δομή του dataset πριν κάνουμε
# οποιαδήποτε επεξεργασία. Ελέγχουμε μέγεθος, στήλες,
# χρονικό εύρος, missing values και κατανομή ομιλιών
# ανά κεντρική τράπεζα.
# ============================================================

# Φόρτωση του dataset
df = pd.read_csv("../data/CBS_dataset_v1.0.csv")

# ── Βασικές πληροφορίες ──────────────────────────────────────
print("=" * 55)
print("  ΒΑΣΙΚΕΣ ΠΛΗΡΟΦΟΡΙΕΣ DATASET")
print("=" * 55)
print(f"  Συνολικές ομιλίες     : {len(df):,}")
print(f"  Αριθμός στηλών        : {len(df.columns)}")
print(f"  Χρονικό εύρος         : {df['Date'].min()} → {df['Date'].max()}")
print(f"  Χώρες                 : {df['Country'].nunique()}")
print(f"  Κεντρικές Τράπεζες    : {df['CentralBank'].nunique()}")

# ── Missing values ───────────────────────────────────────────
print("\n" + "=" * 55)
print("  MISSING VALUES")
print("=" * 55)
print(df.isnull().sum())

# ── Top 10 κεντρικές τράπεζες ─────────────────────────────────
# Κρίσιμο: ελέγχουμε αν η Fed έχει αρκετές ομιλίες
# για να είναι η ανάλυσή μας στατιστικά αξιόπιστη
print("\n" + "=" * 55)
print("  TOP 10 ΚΕΝΤΡΙΚΕΣ ΤΡΑΠΕΖΕΣ (αριθμός ομιλιών)")
print("=" * 55)
print(df['CentralBank'].value_counts().head(10))

# ── Κατανομή γλωσσών ─────────────────────────────────────────
print("\n" + "=" * 55)
print("  ΚΑΤΑΝΟΜΗ ΓΛΩΣΣΩΝ (Top 10)")
print("=" * 55)
print(df['Language'].value_counts().head(10))

  ΒΑΣΙΚΕΣ ΠΛΗΡΟΦΟΡΙΕΣ DATASET
  Συνολικές ομιλίες     : 35,487
  Αριθμός στηλών        : 15
  Χρονικό εύρος         : 1986-01-06 → 2023-12-29
  Χώρες                 : 131
  Κεντρικές Τράπεζες    : 143

  MISSING VALUES
URL               1652
PDF              11019
Title                0
Subtitle         13851
Date                 0
Authorname           0
Role                 0
Gender               0
CentralBank          0
Country              0
text                 0
text_original    30140
Filename             0
Language             0
Source               0
dtype: int64

  TOP 10 ΚΕΝΤΡΙΚΕΣ ΤΡΑΠΕΖΕΣ (αριθμός ομιλιών)
CentralBank
European Central Bank                        2665
Board of Governors of the Federal Reserve    2658
Deutsche Bundesbank                          1418
Bank of England                              1354
Reserve Bank of India                        1071
Bank of Italy                                1035
Central Bank of the Philippines               992
Monetary Auth

# Data Loading and Exploration

In [3]:
df.head()

,URL,PDF,Title,Subtitle,Date,Authorname,Role,Gender,CentralBank,Country,text,text_original,Filename,Language,Source
0,https://www.cbaruba.org/readBlob.do?id=10756,NaN,President speech Managing the Economy as if th...,NaN,2021-12-08,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,Managing the Economy as if the Future Really M...,NaN,abw_10756,English,CB websites
1,https://www.cbaruba.org/readBlob.do?id=7515,NaN,Speech President of the CBA 4th Annual Future ...,NaN,2019-11-01,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,Safeguarding our Future: Strategies for an Aru...,NaN,abw_7515,English,CB websites
2,https://www.cbaruba.org/readBlob.do?id=7518,NaN,Speech Symposium President Semeleer CBA,NaN,2019-09-06,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,FOSTERING ECONOMIC RESILIENCE IN ARUBA; FROM R...,NaN,abw_7518,English,CB websites
3,https://www.cbaruba.org/readBlob.do?id=7548,NaN,Integrity Koninkrijk Seminar,NaN,2016-10-28,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,"Presentation by Mrs Jeanette R. Semeleer, Pres...","Voordracht door mevrouw Jeanette R. Semeleer, ...",abw_7548,Dutch,CB websites
4,https://www.cbaruba.org/readBlob.do?id=7554,NaN,Speech by the President at the BES seminar hel...,NaN,2010-03-29,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,Ongoing changes in the supervisory landscape a...,NaN,abw_7554,English,CB websites


In [4]:
# ============================================================
# CELL 3: Επιλογή Στηλών, Φιλτράρισμα & Language Validation
# ============================================================
# Στόχος: Να κρατήσουμε μόνο τις χρήσιμες στήλες και να
# διασφαλίσουμε ότι δουλεύουμε ΜΟΝΟ με αγγλικά κείμενα.
#
# Σημαντικό: Δεν αρκεί να φιλτράρουμε βάσει της στήλης
# 'Language' — παρατηρήσαμε ότι περιέχει labeling errors
# (π.χ. ομιλία labeled ως 'Dutch' ενώ το κείμενο είναι
# αγγλικό). Κάνουμε language detection απευθείας στο
# κείμενο για 100% αξιοπιστία.
# ============================================================

def detect_language(text):
    """
    Εντοπίζει τη γλώσσα ενός κειμένου χρησιμοποιώντας
    τη βιβλιοθήκη langdetect.
    
    Παράμετροι:
    -----------
    text : str
        Το κείμενο προς ανάλυση.
    
    Επιστρέφει:
    -----------
    str
        Κωδικός γλώσσας (π.χ. 'en', 'fr', 'de') ή
        'unknown' αν δεν είναι δυνατός ο εντοπισμός.
    
    Σημειώσεις:
    -----------
    - Αναλύουμε μόνο τα πρώτα 500 chars για ταχύτητα.
    - Το try/except χειρίζεται κείμενα που είναι πολύ
      κοντά ή έχουν μη αναγνωρίσιμους χαρακτήρες.
    """
    if not isinstance(text, str) or len(text.strip()) < 20:
        return 'unknown'
    try:
        return detect(text[:500])
    except:
        return 'unknown'


# ── Επιλογή χρήσιμων στηλών ──────────────────────────────────
# Κρατάμε μόνο ό,τι χρειαζόμαστε για την ανάλυση:
# Date, text, CentralBank → βασικές για το pipeline
# Country, Authorname     → για sanity check (π.χ. Powell)
# Language, Role          → για φιλτράρισμα και τυχόν ανάλυση
cols = ['Date', 'text', 'CentralBank', 'Country', 'Authorname', 'Language', 'Role']
df = df[cols]

# ── Βήμα 1: Φίλτρο βάσει declared language ───────────────────
# Πρώτο γρήγορο φίλτρο — αφαιρεί τις προφανώς μη-αγγλικές
df_en = df[df['Language'] == 'English'].copy()

# Μετατροπή Date σε datetime — απαραίτητο για χρονοσειρά
df_en['Date'] = pd.to_datetime(df_en['Date'])

# Ταξινόμηση χρονολογικά
df_en = df_en.sort_values('Date').reset_index(drop=True)

# Αφαίρεση γραμμών χωρίς κείμενο
df_en = df_en.dropna(subset=['text'])

# ── Βήμα 2: Language Detection Validation ────────────────────
# Δεύτερο φίλτρο — επαληθεύουμε τη γλώσσα απευθείας
# στο κείμενο για να πιάσουμε τα labeling errors
print("Language detection... (2-3 λεπτά)")
df_en['detected_lang'] = df_en['text'].apply(detect_language)

# ── Αποτελέσματα σύγκρισης ───────────────────────────────────
print("\n" + "=" * 55)
print("  DECLARED vs DETECTED LANGUAGE")
print("=" * 55)
confirmed     = len(df_en[df_en['detected_lang'] == 'en'])
mismatched    = len(df_en[df_en['detected_lang'] != 'en'])
print(f"  Declared English, Detected English : {confirmed:,}")
print(f"  Declared English, Detected OTHER  : {mismatched:,}")
print(f"\n  Top detected non-English:")
print(df_en[df_en['detected_lang'] != 'en']['detected_lang'].value_counts().head(10))

# ── Βήμα 3: Drop non-English ─────────────────────────────────
df_en = df_en[df_en['detected_lang'] == 'en'].copy()

# ── Τελικό summary ────────────────────────────────────────────
print("\n" + "=" * 55)
print("  ΣΥΝΟΨΗ ΦΙΛΤΡΑΡΙΣΜΑΤΟΣ")
print("=" * 55)
print(f"  Αρχικές ομιλίες                   : {len(df):,}")
print(f"  Μετά φίλτρο 'English' label        : {confirmed + mismatched:,}")
print(f"  Μετά language detection validation : {len(df_en):,}")
print(f"  Χρονικό εύρος                      : "
      f"{df_en['Date'].min().date()} → {df_en['Date'].max().date()}")

Language detection... (2-3 λεπτά)

  DECLARED vs DETECTED LANGUAGE
  Declared English, Detected English : 30,058
  Declared English, Detected OTHER  : 82

  Top detected non-English:
detected_lang
fr    31
de    21
id    18
it     3
nl     3
lt     2
tl     1
es     1
ca     1
pt     1
Name: count, dtype: int64

  ΣΥΝΟΨΗ ΦΙΛΤΡΑΡΙΣΜΑΤΟΣ
  Αρχικές ομιλίες                   : 35,487
  Μετά φίλτρο 'English' label        : 30,140
  Μετά language detection validation : 30,058
  Χρονικό εύρος                      : 1986-01-06 → 2023-12-28


In [ ]:
# ============================================================
# CELL 4: Καθάρισμα Κειμένου & Tokenization
# ============================================================
# Στόχος: Να μετατρέψουμε κάθε ομιλία από raw text σε
# καθαρή λίστα tokens έτοιμη για ανάλυση.
#
# Επιστρέφουμε ΛΙΣΤΑ (όχι string) γιατί στο επόμενο βήμα
# χρειαζόμαστε πρόσβαση σε συγκεκριμένες θέσεις tokens
# για το negation handling (π.χ. "not hawkish").
# ============================================================

# Φόρτωση stopwords
stop_words = set(stopwords.words('english'))

def clean_and_tokenize(text):
    """
    Καθαρίζει κείμενο και επιστρέφει λίστα tokens
    χωρίς stopwords.
    
    Παράμετροι:
    -----------
    text : str
        Raw κείμενο ομιλίας.
    
    Επιστρέφει:
    -----------
    list
        Λίστα καθαρών tokens χωρίς stopwords.
        Επιστρέφει κενή λίστα αν το input δεν είναι string.
    
    Βήματα επεξεργασίας:
    --------------------
    1. Πεζά γράμματα       → "Inflation" == "inflation"
    2. Αφαίρεση αριθμών    → δεν έχουν σημασιολογική αξία
    3. Αφαίρεση στίξης     → κρατάμε μόνο λέξεις
    4. Tokenization        → split σε λίστα λέξεων
    5. Αφαίρεση stopwords  → αφαιρούμε "the", "is", "a"...
    
    Σημείωση: Επιστρέφουμε λίστα (όχι string) για να
    μπορούμε να ελέγχουμε συγκεκριμένες θέσεις στο
    negation handling του επόμενου cell.
    """
    if not isinstance(text, str):
        return []
    
    # Βήμα 1: Πεζά
    text = text.lower()
    # Βήμα 2: Αφαίρεση αριθμών
    text = re.sub(r'\d+', '', text)
    # Βήμα 3: Αφαίρεση σημείων στίξης — κρατάμε μόνο γράμματα και κενά
    text = re.sub(r'[^\w\s]', '', text)
    # Βήμα 4: Tokenization
    tokens = text.split()
    # Βήμα 5: Αφαίρεση stopwords
    tokens = [t for t in tokens if t not in stop_words]
    
    return tokens


# ── Εφαρμογή στο dataset ─────────────────────────────────────
print("Καθάρισμα κειμένων... (1-2 λεπτά)")
df_en['tokens'] = df_en['text'].apply(clean_and_tokenize)

# Αριθμός λέξεων ΜΕΤΑ αφαίρεση stopwords — αυτός θα
# χρησιμοποιηθεί ως παρονομαστής στον υπολογισμό του score
df_en['word_count'] = df_en['tokens'].apply(len)

# ── Στατιστικά κειμένων ──────────────────────────────────────
print("\n" + "=" * 55)
print("  ΣΤΑΤΙΣΤΙΚΑ ΚΕΙΜΕΝΩΝ (μετά καθάρισμα)")
print("=" * 55)
print(f"  Μέσος αριθμός λέξεων  : {df_en['word_count'].mean():.0f}")
print(f"  Διάμεσος λέξεων       : {df_en['word_count'].median():.0f}")
print(f"  Ελάχιστο              : {df_en['word_count'].min()}")
print(f"  Μέγιστο               : {df_en['word_count'].max():,}")

# ── Φίλτρο ελάχιστου μήκους ─────────────────────────────────
# Αφαιρούμε ομιλίες < 50 λέξεων — πολύ κοντά κείμενα
# (π.χ. press releases) δεν αντιπροσωπεύουν πραγματική
# νομισματική επικοινωνία και θα έδιναν αναξιόπιστα scores
before = len(df_en)
df_en = df_en[df_en['word_count'] >= 50].reset_index(drop=True)
after  = len(df_en)

print("\n" + "=" * 55)
print("  ΦΙΛΤΡΟ ΕΛΑΧΙΣΤΟΥ ΜΗΚΟΥΣ (>= 50 λέξεις)")
print("=" * 55)
print(f"  Αφαιρέθηκαν (< 50 λέξεις) : {before - after:,}")
print(f"  Τελικές ομιλίες για ανάλυση : {after:,}")

Καθάρισμα κειμένων... (1-2 λεπτά)
